In [4]:
from constants import *
from data_utils import *
from generate import *
from scipy import stats
import pybedtools

In [ ]:
def artifacts_stats():
    gene_locs = load_file('gene_locations.csv', local_dir=DATA_DIR)
    zscore_changes_by_arm = load_file('single_cell_ko_arm_zscores.csv', local_dir=PROCESSED_DIR, index_col=[0, 1, 2], header=[0])
    ko_metadata = load_file('knockout_metadata.csv', local_dir=METADATA_DIR)
    controls = sorted(ko_metadata[ko_metadata['target_class'] == 'Olfactory Control']['knockout'].unique().tolist())

    arm_zscore_longform = zscore_changes_by_arm.loc[pd.IndexSlice[:, controls, 'all']].groupby(level=['assigned_ko']).mean().melt(ignore_index=False).reset_index().merge(
        zscore_changes_by_arm.loc[pd.IndexSlice[:, controls, 'normal']].groupby(level=['assigned_ko']).mean().melt(ignore_index=False).reset_index(), 
        left_on=['assigned_ko', 'variable'], right_on=['assigned_ko', 'variable']
    ).rename({'value_x': 'all', 'value_y': 'normal', 'variable': 'arm'}, axis=1)
    arm_zscore_longform = arm_zscore_longform.merge(gene_locs[['Gene name', 'ChrArm']], left_on='assigned_ko', right_on='Gene name')
    arm_zscore_longform['Targeted arm'] = (arm_zscore_longform['ChrArm'] == arm_zscore_longform['arm']).astype(str)

    print('Across all control knockouts in all cell lines, we found decreased expression at the targeted arm')
    print('median z-scored expression target arm:', arm_zscore_longform[arm_zscore_longform['Targeted arm'] == 'True']['all'].median())
    print('median z-scored expression target arm after removal of cells with aberrant position-dependent arm expression:',
        arm_zscore_longform[arm_zscore_longform['Targeted arm'] == 'True']['normal'].median())

    min_cells=50
    arm_alteration_covariates = load_file('arm_alteration_covariates.csv', local_dir=PROCESSED_DIR)
    vector_comparison_table = arm_alteration_covariates[arm_alteration_covariates['Total cells per KO'] >= min_cells]
    xlabel='v1_arm_trunc_freq'
    ylabel='v2_arm_trunc_freq'
    corr_result = scipy.stats.pearsonr(vector_comparison_table[xlabel], vector_comparison_table[ylabel])

    print('\nconsistency of arm truncation frequency between guides targeting the same gene:')
    print(corr_result, f'n={len(vector_comparison_table)}')

    arm_alteration_summary_table = load_file('arm_alteration_covariates.csv', local_dir=PROCESSED_DIR)
    arm_alteration_correlations = load_file('arm_alteration_correlations.csv', local_dir=PROCESSED_DIR)
    arm_alteration_subset = arm_alteration_correlations[arm_alteration_correlations['Outcome'] == 'Arm truncation frequency']
    arm_alteration_subset = arm_alteration_subset.merge(arm_alteration_summary_table.groupby('cell_line')['Total cells per KO'].mean(), left_on='Cell line', right_index=True)
    arm_avg_df = arm_alteration_subset.query('Predictor == "Average arm dependency"')
    relation_df = arm_avg_df[arm_avg_df['Correlation with outcome'] < -.25]
    other_min = arm_avg_df[~arm_avg_df['Cell line'].isin(relation_df['Cell line'])]['Correlation with outcome'].min()
    other_max = arm_avg_df[~arm_avg_df['Cell line'].isin(relation_df['Cell line'])]['Correlation with outcome'].max()

    print('\nn cell lines with modest relationship between arm loss and avg arm dependency:', len(relation_df))
    print('corr between -.25 and ', relation_df['Correlation with outcome'].min())
    print('aggregate pearson r =',arm_avg_df['Aggregate correlation with outcome'].unique(), 'n=', len(arm_alteration_summary_table))
    print(f'range remaning models: {other_min}, {other_max}')

    arm_alteration_covariates = load_file('arm_alteration_covariates.csv', local_dir=PROCESSED_DIR)
    cl_metadata = load_file('cell_line_metadata.csv', local_dir=METADATA_DIR)
    tp53_damaging = (load_file('OmicsSomaticMutationsMatrixDamaging', local_dir=DOWNLOADED_DIR, index_col=0).reindex(
        index=cl_metadata['arxspan_id'].tolist())['TP53 (7157)'] > 0).rename('TP53 status').replace({True: 'Mutant', False: 'WT'})
    arm_alteration_covariates = arm_alteration_covariates.merge(cl_metadata.merge(tp53_damaging, left_on='arxspan_id', right_index=True)[['cell_line', 'TP53 status']])

    print('\nMedian truncation frequency by TP53 status:')
    print(arm_alteration_covariates.groupby(['cell_line', 'TP53 status'])['Arm truncation frequency'].median().groupby(level=1).median())

    mwu_result = scipy.stats.mannwhitneyu(
        arm_alteration_covariates[(arm_alteration_covariates['TP53 status'] == 'WT')].groupby('cell_line')['Arm truncation frequency'].median().tolist(), 
        arm_alteration_covariates[(arm_alteration_covariates['TP53 status'] == 'Mutant')].groupby('cell_line')['Arm truncation frequency'].median().tolist()
    )
    print('Mann-Whitney result on medians: TP53 mut. vs. WT')
    print(mwu_result)

    print('\narm gain median frequency:', arm_alteration_summary_table['Arm gain frequency'].median())
    print('arm loss median frequency:', arm_alteration_summary_table['Arm truncation frequency'].median())
    top_cell_line, top_predictor = arm_alteration_correlations[
        arm_alteration_correlations['Outcome'] == 'Arm gain frequency'
    ].sort_values('Correlation with outcome').iloc[0, :].loc[['Cell line', 'Predictor']]
    subset = arm_alteration_summary_table[[top_predictor, 'Arm gain frequency']].dropna()
    full_corr_res = scipy.stats.pearsonr(subset[top_predictor], subset['Arm gain frequency'])
    print(f'{top_predictor} correlation:', full_corr_res, 'n=', len(arm_alteration_summary_table))

    df = arm_alteration_correlations[arm_alteration_correlations['Predictor'] == 'Arm gain frequency']
    ls = df[df['Correlation with outcome'] > .3]['Cell line'].unique()
    print('lines with r<.03 between arm gain and loss:', ls)

    cell_lines = load_file('cell_line_metadata.csv', local_dir=METADATA_DIR)['cell_line'].tolist()
    test_g = 'SGO1-AS1'
    test_g_df = pd.DataFrame()
    for cl in tqdm(cell_lines):
        cl_to_sceptre_test = load_file('discovery_analysis.csv', local_dir=os.path.join(PROCESSED_DIR, 'sceptre', 'arm_trunc', cl))
        cl_to_sceptre_test['cell_line'] = cl
        df = cl_to_sceptre_test.query(f'response_id == "{test_g}"')
        test_g_df = pd.concat([test_g_df, df])

    gain_z = test_g_df.query('grna_target == "GAIN"')['z_orig'].dropna().values
    trunc_z = test_g_df.query('grna_target == "TRUNC"')['z_orig'].dropna().values
    t_stat, p_value = stats.ttest_1samp(trunc_z, gain_z)
    print('SGO1-AS1 lower in gain, higher in loss pval=', p_value)


    cl_to_arxspan_id = cl_metadata.set_index('cell_line')['arxspan_id'].to_dict()
    arxspan_id_to_cl = cl_metadata.set_index('arxspan_id')['cell_line'].to_dict()
    cell_lines = cl_metadata.cell_line.to_list()

    all_cells_table = load_file('all_cells_table.csv', local_dir=PROCESSED_DIR)

    cell_calling_df = pd.DataFrame({
        'Passing\nsinglet': all_cells_table[all_cells_table['pass_qc'] & ~all_cells_table['arm_trunc']].value_counts('cell_line'),
        'Singlet with\narm loss': all_cells_table[all_cells_table['pass_qc'] & all_cells_table['arm_trunc']].value_counts('cell_line'),
        'Multiple\ninfection': all_cells_table.groupby('cell_line').apply(lambda x: len(x[
            (x['grna_max_umi'] >= 5) & ((x['grna_max_umi'] / x['grna_n_umis']) < 0.8) & 
            (x['response_n_umis'] >= x['response_n_umis'].quantile(0.01)) & (x['response_n_umis'] <= x['response_n_umis'].quantile(0.99)) &
            (x['response_n_nonzero'] >= x['response_n_nonzero'].quantile(0.01)) & (x['response_n_nonzero'] <= x['response_n_nonzero'].quantile(0.99)) &
            (x['response_p_mito'] <= 0.25)
        ])),
        'Ambiguous\ninfection': all_cells_table.groupby('cell_line').apply(lambda x: len(x[
            (x['grna_max_umi'] > 1) & (x['grna_max_umi'] < 5) &
            (x['response_n_umis'] >= x['response_n_umis'].quantile(0.01)) & (x['response_n_umis'] <= x['response_n_umis'].quantile(0.99)) &
            (x['response_n_nonzero'] >= x['response_n_nonzero'].quantile(0.01)) & (x['response_n_nonzero'] <= x['response_n_nonzero'].quantile(0.99)) &
            (x['response_p_mito'] <= 0.25)
        ])),
        'Unperturbed': all_cells_table.groupby('cell_line').apply(lambda x: len(x[
            (x['grna_max_umi'] <= 1) & 
            (x['response_n_umis'] >= x['response_n_umis'].quantile(0.01)) & (x['response_n_umis'] <= x['response_n_umis'].quantile(0.99)) &
            (x['response_n_nonzero'] >= x['response_n_nonzero'].quantile(0.01)) & (x['response_n_nonzero'] <= x['response_n_nonzero'].quantile(0.99)) &
            (x['response_p_mito'] <= 0.25)
        ])),
        'Other\nQC fail': all_cells_table.groupby('cell_line').apply(lambda x: len(x[
            (x['response_n_umis'] < x['response_n_umis'].quantile(0.01)) | (x['response_n_umis'] > x['response_n_umis'].quantile(0.99)) |
            (x['response_n_nonzero'] < x['response_n_nonzero'].quantile(0.01)) | (x['response_n_nonzero'] > x['response_n_nonzero'].quantile(0.99)) |
            (x['response_p_mito'] > 0.25)
        ]))
    })

    print('\n')
    df = (cell_calling_df.T / all_cells_table.value_counts('cell_line')).T
    for col in df.columns:
        vals = df[col]
        print(f'{col} percent range: {vals.min()}, {vals.max()}')

    vals = df['Multiple\ninfection'] + df['Ambiguous\ninfection']
    print(f'ambiguous or mixed percent range: {vals.min()}, {vals.max()}')

    vals = cell_calling_df['Singlet with\narm loss'] / cell_calling_df['Passing\nsinglet']
    print(f'pct arm abberation of cells otherwise passing QC: {vals.min()}, {vals.max()}')
    print('median =', vals.median())

def heterogeneity_stats():
    all_genesets = load_file('all_genesets.csv', local_dir=DATA_DIR)
    control_single_cell_gene_stats, median_single_cell_stats, high_var_common_genes, high_expr_excluded = select_high_variance_genes(
        load_file('single_cell_control_gene_stats.csv', local_dir=PROCESSED_DIR)
    )
    selected = control_single_cell_gene_stats[control_single_cell_gene_stats['selected']]['gene'].unique().tolist()

    mitochondrial_genes = all_genesets[(all_genesets['collection'] == 'HGNC') & (all_genesets['term'].str.contains('Mitochondrially encoded'))]['gene'].tolist()
    ribosomal_genes = all_genesets[(all_genesets['collection'] == 'HGNC') & (all_genesets['term'].str.contains('(R|r)ibosom'))]['gene'].tolist()
    housekeeping_genes = all_genesets[all_genesets['term'] == 'HSIAO_HOUSEKEEPING_GENES']['gene'].tolist()

    high_variance_stats_df = high_var_common_genes.copy()
    high_variance_stats_df['highly_expressed'] = ~high_variance_stats_df.index.isin(high_expr_excluded.index.tolist())
    high_variance_stats_df['mitochondrial'] = high_variance_stats_df.index.isin(mitochondrial_genes)
    high_variance_stats_df['ribosomal'] = high_variance_stats_df.index.isin(ribosomal_genes)
    high_variance_stats_df['housekeeping'] = high_variance_stats_df.index.isin(housekeeping_genes)
    high_variance_stats_df['selected'] = high_variance_stats_df.index.isin(selected)

    print('Highly variable genes:', high_variance_stats_df.shape[0])
    print('Highly expressed genes:', high_variance_stats_df['highly_expressed'].sum())
    print('Lowly expressed genes:', (~high_variance_stats_df['highly_expressed']).sum())
    print('Mitochondrial genes:', high_variance_stats_df['mitochondrial'].sum())
    print('Ribosomal genes:', high_variance_stats_df['ribosomal'].sum())
    print('Housekeeping genes:', high_variance_stats_df['housekeeping'].sum())

    g = 'S100A8'
    top_variance_expression = load_file('single_cell_high_variance_raw_expression.csv', local_dir=PROCESSED_DIR)
    top_var_gene_expr_subset = top_variance_expression.dropna(subset=[g])
    print('\n', g)
    print('Nonzero fraction of cells:', top_var_gene_expr_subset.assign(nonzero=lambda x: x[g] > 0).groupby('cell_line')['nonzero'].mean().rename_axis(''))
    print('total:', top_var_gene_expr_subset.assign(nonzero=lambda x: x[g] > 0)['nonzero'].mean())

def qc_stats():
    pseudobulk_correlation_with_ccle = load_file('pseudobulk_correlation_with_ccle.csv', local_dir=PROCESSED_DIR)

    # SLR23 does not have expression data in DepMap 25Q2
    pseudobulk_correlation_with_ccle = pseudobulk_correlation_with_ccle[pseudobulk_correlation_with_ccle['scCellLine'] != 'SLR23']

    print('is negative control pseudobulk profile most correlated?:')
    for cl in pseudobulk_correlation_with_ccle['scCellLine'].unique():
        cl_df = pseudobulk_correlation_with_ccle[pseudobulk_correlation_with_ccle['scCellLine'] == cl]
        print(cl, cl_df.loc[cl_df['TopVarCorrelation'].idxmax()]['Identity'])

    matching_cor = pseudobulk_correlation_with_ccle[pseudobulk_correlation_with_ccle['Identity']]['TopVarCorrelation']
    mismatching_cor = pseudobulk_correlation_with_ccle[~pseudobulk_correlation_with_ccle['Identity']]['TopVarCorrelation']

    print('cor range in matching cell lines: ', matching_cor.min(), matching_cor.max())
    print('cor range in mismatching cell lines: ', mismatching_cor.min(), mismatching_cor.max())

    cells_per_ko_table = load_file('depletion_vs_crispr_table.csv', local_dir=PROCESSED_DIR)
    corr_result = stats.pearsonr(cells_per_ko_table.dropna()["depletion"], cells_per_ko_table.dropna()["gene_effect"])
    print('\ncorrelation between cell depletion and DepMap gene effect scores irrespective of cell line identity')
    print(corr_result, 'n=', len(cells_per_ko_table))

    cl_cors = dict()
    for cl in cells_per_ko_table['cell_line'].unique():
        cl_df = cells_per_ko_table[cells_per_ko_table['cell_line'] == cl]
        cl_cors[cl] = stats.pearsonr(cl_df.dropna()["depletion"], cl_df.dropna()["gene_effect"]).statistic
    cl_cors = pd.Series(cl_cors)
    print('range cor within cell line: ', cl_cors.min(), cl_cors.max())
    print('median cor within cell line: ', cl_cors.median())

    cell_quality_covariates = load_file('cell_quality_covariates.csv', local_dir=PROCESSED_DIR)
    cl_metadata = load_file('cell_line_metadata.csv', local_dir=METADATA_DIR)
    cell_quality_covariates = cell_quality_covariates.merge(cl_metadata.set_index('cell_line')['OncotreeLineage'], left_on='cell_line', right_index=True)

    corr_result = stats.pearsonr(cell_quality_covariates["total_umis_per_ko"], cell_quality_covariates["n_self_downregulated"])
    print('\nthe number of significantly downregulated targets cor with total number of UMIs recovered across all cells with the perturbation')
    print('correlation:', corr_result, 'n=', len(cell_quality_covariates))

    xlabels = ['avg_umis_per_ko', 'n_cells_per_ko', 'avg_genes_per_ko']
    xprettys = ['Mean UMIs per cell per knockout', 'Mean cells per knockout', 'Mean genes per cell per knockout']
    ylabel = 'n_self_downregulated'
    ypretty = 'sgRNA targets downregulated'

    print('\n')
    for i, xlab in enumerate(xlabels):
        corr_result = stats.pearsonr(cell_quality_covariates[xlab], cell_quality_covariates[ylabel])
        print(f'{xprettys[i]} correlation:', corr_result)

    myc_df = load_file('myc_validation_enrichment_table.csv', local_dir=PROCESSED_DIR, index_col=0)
    cl_medians = {}
    for cl in myc_df['cell_line'].unique():
        cl_medians[cl] = myc_df[myc_df['cell_line'] == cl]['z'].median()
    cl_medians = pd.Series(cl_medians)
    print('\nz-scored change in expression of MYC targets following MYC knockout, finding broad downregulation in all cell lines')
    print('max median z-score per cell line:', cl_medians.max())

def global_response_stats():
    transcriptional_change_df = load_file('transcriptional_change_table.csv', local_dir=PROCESSED_DIR)
    transcriptional_change_df = transcriptional_change_df.dropna(subset=['deviation_from_basal', 'gene_effect'])

    corr_result = stats.pearsonr(transcriptional_change_df["gene_effect"], transcriptional_change_df["deviation_from_basal"])
    print('observed stronger deviations from a cell line’s basal transcription state as the strength of the dependency increased')
    print('correlation:', corr_result, 'n=', len(transcriptional_change_df))

    top_dependency_diff_expr_gene_df = load_file('dependency_top_diff_expr_gene_table.csv', local_dir=PROCESSED_DIR)
    expr_order = top_dependency_diff_expr_gene_df.set_index('response_id')['response_id_order'].drop_duplicates().sort_values().index.tolist()
    top_deg_corrs = fast_cor(top_dependency_diff_expr_gene_df.pivot(index='grna_target', columns='response_id', values='mean_dep_z')).loc[expr_order, expr_order]
    up_corrs = top_deg_corrs.loc[expr_order[:10], expr_order[:10]]
    down_corrs = top_deg_corrs.loc[expr_order[10:], expr_order[10:]]
    up_corrs_flat = up_corrs.values[(np.triu(np.ones_like(up_corrs), k=1) == 1)]
    down_corrs_flat = down_corrs.values[(np.triu(np.ones_like(down_corrs), k=1) == 1)]
    print('\nupreg median correlation:', np.quantile(up_corrs_flat, q=0.25))
    print('downreg median correlation:', np.quantile(down_corrs_flat, q=0.25))
    print('downreg 10th percentile cor', np.percentile(down_corrs_flat, 10))
    print('upreg 10th percentile cor', np.percentile(up_corrs_flat, 10))

    dependent_hallmark_mean_z = load_file('dependency_hallmark_mean_z_matrix.csv', local_dir=PROCESSED_DIR)
    dependent_hallmark_mean_z = dependent_hallmark_mean_z.set_index('term')
    print('\nn dependent knockouts =', dependent_hallmark_mean_z.shape[1])
    hallmark_cor = fast_cor(dependent_hallmark_mean_z.T)
    print(hallmark_cor.loc[['G2M CHECKPOINT', 'E2F TARGETS'], ['MYC TARGETS V1', 'MYC TARGETS V2']])

    geneset_zscore_matrix = load_file('geneset_mean_zscore_matrix.csv', local_dir=PROCESSED_DIR, header=[0, 1], index_col=[0, 1])
    crispr_table = load_file('crispr_table.csv', local_dir=PROCESSED_DIR)

    selected_genesets = [
        'HALLMARK_E2F_TARGETS',
        'HALLMARK_G2M_CHECKPOINT',
        'HALLMARK_MYC_TARGETS_V1',
        'HALLMARK_MYC_TARGETS_V2',
        'HALLMARK_UNFOLDED_PROTEIN_RESPONSE',
        'HALLMARK_OXIDATIVE_PHOSPHORYLATION',
        'HALLMARK_GLYCOLYSIS',
        'HALLMARK_INTERFERON_ALPHA_RESPONSE',
        'HALLMARK_INTERFERON_GAMMA_RESPONSE',
        'HALLMARK_P53_PATHWAY',
        'HALLMARK_APOPTOSIS',
        'HALLMARK_TNFA_SIGNALING_VIA_NFKB'
    ]

    dependent_geneset_z_matrix = geneset_zscore_matrix.loc[
        pd.IndexSlice['Hallmark', selected_genesets], 
        crispr_table[crispr_table['is_dependent']].apply(lambda x: (x['cell_line'], x['guide']), axis=1).values.tolist()
    ].droplevel(0, axis=0)
    dependent_geneset_mean_z_matrix = dependent_geneset_z_matrix.T.groupby(level=1).mean().T

    ko_order = dependent_geneset_mean_z_matrix.loc[
        ['HALLMARK_E2F_TARGETS', 'HALLMARK_G2M_CHECKPOINT', 'HALLMARK_MYC_TARGETS_V1', 'HALLMARK_MYC_TARGETS_V2'], 
        # ['HALLMARK_E2F_TARGETS', 'HALLMARK_G2M_CHECKPOINT'], 
        :
    ].mean().sort_values().index.tolist()
    dependent_geneset_mean_z_matrix.index = dependent_geneset_mean_z_matrix.index.map(lambda x: clean_geneset_name(x, nth=10, remove_prefix=True))

    print('\nmodest upregulation of apoptotic pathways across most dependent knockouts')
    print('median z-score =\n', dependent_geneset_mean_z_matrix.loc[['APOPTOSIS', 'P53 PATHWAY', 'TNFA SIGNALING VIA NFKB']].T.median())

    print('\ninterferon pathways and the apoptotic pathways showed correlated responses across dependent knockouts')
    r = hallmark_cor.loc[['INTERFERON ALPHA RESPONSE', 'INTERFERON GAMMA RESPONSE'], 
                        ['APOPTOSIS', 'P53 PATHWAY', 'TNFA SIGNALING VIA NFKB']].mean().mean()
    print('mean pearson r =', r)

    print('\nunfolded protein response and oxidative phosphorylation exhibited knockout-specific responses uncorrelated with apoptotic signatures')
    r = hallmark_cor.loc[['UNFOLDED PROTEIN RESPONSE', 'OXIDATIVE PHOSPHORYLATION'], 
                    ['APOPTOSIS', 'P53 PATHWAY', 'TNFA SIGNALING VIA NFKB']].mean(axis=1)
    print('mean pearson r =\n', r)

    print('\nunfolded protein response and oxidative phosphorylation modestly correlated with cell proliferation')
    r = hallmark_cor.loc[['UNFOLDED PROTEIN RESPONSE', 'OXIDATIVE PHOSPHORYLATION'], 
                    ['E2F TARGETS', 'G2M CHECKPOINT']].mean(axis=1)
    print('mean pearson r =\n', r)

    targets_to_expand=['PITRM1', 'NHLRC2', 'MTPAP', 'PRKRA', 'RNF31', 'SUZ12', 'SLC25A3']
    fdr_threshold=0.05
    n_top_genes=None
    out_dir = None

    all_genesets = load_file('all_genesets.csv', local_dir=DATA_DIR)
    cl_metadata = load_file('cell_line_metadata.csv', local_dir=METADATA_DIR)
    sceptre_zscore = load_file('zscore_matrix.csv', local_dir=PROCESSED_DIR, index_col=0, header=[0, 1]).rename_axis(['cell_line', 'grna_target'], axis=1)
    sceptre_fdr = load_file('fdr_matrix.csv', local_dir=PROCESSED_DIR, index_col=0, header=[0, 1]).rename_axis(['cell_line', 'grna_target'], axis=1)

    dependencies_to_expand = crispr_table[crispr_table['is_dependent'] & crispr_table['guide'].isin(targets_to_expand)].set_index(['cell_line', 'guide']).index.tolist()

    significant_genes = (sceptre_fdr.loc[:, dependencies_to_expand] < fdr_threshold).apply(lambda x: x.loc[lambda y: y == True].index.tolist())
    gene_universes = sceptre_fdr.apply(lambda x: x.dropna().index.tolist())
    gsea_enrichments = pd.concat({
        x: run_multiple_hypergeometric(
            query_genes=significant_genes.loc[x],
            geneset_table=all_genesets[(all_genesets['collection'] == 'GO:BP') & (all_genesets['original_set_size'] >= 25)],
            all_genes=gene_universes.loc[x],
            report_genes=True
        ) for x in dependencies_to_expand if len(significant_genes.loc[x]) >= 2
    }).droplevel(2, axis=0).rename_axis(['cell_line', 'assigned_ko']).reset_index()
    if out_dir is not None:
        gsea_enrichments.to_csv(os.path.join(out_dir, 'mistimed_perturbation_enrichment_table.csv'), index=False)

    genesets_to_highlight = gsea_enrichments[gsea_enrichments['FDR'] < fdr_threshold].sort_values('FDR').groupby('assigned_ko').head(1)['term'].unique().tolist()
    # manual palette assignment
    geneset_to_color = {'GOBP_OXIDATIVE_PHOSPHORYLATION': 'tab:blue',
                        'GOBP_CELLULAR_RESPONSE_TO_MOLECULE_OF_BACTERIAL_ORIGIN': 'tab:olive',
                        'GOBP_ATP_SYNTHESIS_COUPLED_ELECTRON_TRANSPORT': 'tab:blue',
                        'GOBP_REGULATION_OF_PROGRAMMED_CELL_DEATH': 'tab:red'}
    genesets_to_gene = {gs: all_genesets[all_genesets['term'] == gs]['gene'].tolist() for gs in genesets_to_highlight}
    print('significant genesets: ', genesets_to_highlight)
    print('palette: ', geneset_to_color)

    expr_genes_to_show_df = pd.DataFrame({
        'perturbations_significant': (sceptre_fdr.loc[:, dependencies_to_expand] < fdr_threshold).sum(axis=1),
        'significant_abs_z': (sceptre_zscore[sceptre_fdr < fdr_threshold].loc[:, dependencies_to_expand]).abs().max(axis=1)
    })
    expr_genes_to_show = expr_genes_to_show_df[expr_genes_to_show_df['perturbations_significant'] >= 1].sort_values('significant_abs_z', ascending=False).head(n_top_genes).index.tolist()

    mtx = sceptre_zscore.loc[expr_genes_to_show, dependencies_to_expand].fillna(0)
    cg = sns.clustermap(
        mtx, cmap='RdBu_r', vmin=-5, center=0, vmax=5, method='ward'
    )
    plt.close()

    print('\n')
    expr_order = cg.data2d.index.tolist()
    ko_to_cls = dict()
    ko_to_heatmap = dict()
    for ko in targets_to_expand:
        ko_to_heatmap[ko] = cg.data2d.loc[:, pd.IndexSlice[:, ko]]
        ko_to_cls[ko] = ko_to_heatmap[ko].droplevel(level=1, axis=1).columns.tolist()

    print('\nmost enriched gene sets covered oxidative phosphorylation in MTPAP knockout')
    print(gsea_enrichments.query('assigned_ko == "MTPAP" & term == "GOBP_OXIDATIVE_PHOSPHORYLATION"')['FDR'].min())

    print('\nimmune response in RNF31 knockout')
    print(gsea_enrichments.query('assigned_ko == "RNF31" & term == "GOBP_CELLULAR_RESPONSE_TO_MOLECULE_OF_BACTERIAL_ORIGIN"')['FDR'])

    print('\napoptosis in SUZ12 knockout')
    print(gsea_enrichments.query('assigned_ko == "SUZ12" & term == "GOBP_REGULATION_OF_PROGRAMMED_CELL_DEATH"')['FDR'])

    print('\nExpected cell recovery relative to control cells by regression compared to observed cell recovery for dependent perturbations')
    timepoint_pred_table = load_file('timepoint_prediction_table.csv', local_dir=PROCESSED_DIR)
    print('N=', len(timepoint_pred_table))

def common_response_stats():
    full_corr_matrix = load_file('full_corr_matrix.csv', local_dir=PROCESSED_DIR, header=[0, 1], index_col=[0, 1])
    sorted_full_corr_matrix = full_corr_matrix.dropna(how='all', axis=0).dropna(how='all', axis=1).sort_index(level=[1, 0], axis=0).sort_index(level=[1, 0], axis=1)

    print('defined a network of gene targets using pairwise correlations \nbetween Z-score change in expression across our entire \nintegrated dataset spanning 16 cell lines and 90 test genes, \nexcluding perturbations for which we recovered fewer than 5 cells')
    print('n=', len(sorted_full_corr_matrix))

    unique_correlation_combinations = load_file('unique_perturbation_pairs.csv', local_dir=PROCESSED_DIR)
    ko_pair_summary = load_file('perturbation_pair_correlation_summary.csv', local_dir=PROCESSED_DIR)
    ko_pair_summary['all_cross_lines'] = ko_pair_summary['all_cross_lines_medium']
    graph_subset_df = ko_pair_summary[ko_pair_summary['n_pairs_high_corr'] >= 4]
    graph_subset_df.loc[graph_subset_df['grna_target1'] == graph_subset_df['grna_target2'], 'all_cross_lines'] = False

    print('\nXRN1 and SMG6 showed the highest degree of similarity across all pairs of cell lines')
    cor = unique_correlation_combinations.query('grna_target1 == "SMG6" & grna_target2 == "XRN1"')['r']
    print('cor range:', cor.min(), cor.max())
    print('median', cor.median())

    print('\nADAR knockout inducing a similar response with XRN1 and SMG6')
    xrn1_cor = unique_correlation_combinations.query('grna_target1 == "ADAR" & grna_target2 == "XRN1"')['r'].median()
    smg6_cor = unique_correlation_combinations.query('grna_target1 == "ADAR" & grna_target2 == "SMG6"')['r'].median()
    print('XRN1 median r', xrn1_cor)
    print('SMG6 median r', smg6_cor)

    all_genesets = load_file('all_genesets.csv', local_dir=DATA_DIR)
    sceptre_zscore = load_file('zscore_matrix.csv', local_dir=PROCESSED_DIR, index_col=0, header=[0, 1]).rename_axis(['cell_line', 'grna_target'], axis=1)
    sceptre_fdr = load_file('fdr_matrix.csv', local_dir=PROCESSED_DIR, index_col=0, header=[0, 1]).rename_axis(['cell_line', 'grna_target'], axis=1)

    print('\nXRN1 and SMG6 perturbations induced an extreme upregulation of small nucleolar RNAs')
    snornas = all_genesets[all_genesets['term'] == 'Small nucleolar RNA non-coding host genes']['gene'].tolist()
    print((sceptre_zscore.groupby(level=1, axis=1).mean().loc[:, ['SMG6', 'XRN1', 'ADAR']].reindex(index=snornas) > 5).sum().rename('# snoRNAs with z-score > 5'))

    grna_target='TFRC'
    top_n=10
    min_lines = 8
    fdr_threshold=0.05

    cl_metadata = load_file('cell_line_metadata.csv', local_dir=METADATA_DIR)
    crispr_table = load_file('crispr_table.csv', local_dir=PROCESSED_DIR)

    z_long = sceptre_zscore.loc[:, pd.IndexSlice[:, 'TFRC']].melt(ignore_index=False, value_name='zscore').reset_index()
    fdr_long = sceptre_fdr.loc[:, pd.IndexSlice[:, 'TFRC']].melt(ignore_index=False, value_name='FDR').reset_index()

    longform_sceptre_results = pd.concat([
        z_long, fdr_long['FDR']
    ], axis=1).dropna(subset=['zscore'])
    longform_sceptre_results = longform_sceptre_results.merge(
        crispr_table.rename({'guide': 'grna_target'}, axis=1)[['cell_line', 'grna_target', 'gene_effect']]
    )
    longform_sceptre_results['abs_z'] = longform_sceptre_results['zscore'].abs()
    rank_by_perturbation = longform_sceptre_results[longform_sceptre_results['FDR'] < 0.05].groupby(['cell_line', 'grna_target'])['abs_z'].rank(ascending=False)
    longform_sceptre_results['significant_rank'] = rank_by_perturbation
    longform_sceptre_results = longform_sceptre_results.merge(
        longform_sceptre_results.value_counts(['response_id', 'grna_target']).rename('response_id_n_cell_lines').reset_index()
    )
    longform_sceptre_results['lineage'] = longform_sceptre_results['cell_line'].map(cl_metadata.set_index('cell_line')['OncotreeLineage'])
    longform_sceptre_results = longform_sceptre_results.merge(
        longform_sceptre_results[longform_sceptre_results['FDR'] < fdr_threshold].value_counts(['grna_target', 'response_id']).rename('response_id_n_significant_cell_lines').reset_index(),
        how='left')

    all_results_df = longform_sceptre_results
    result_subset = all_results_df[
                (all_results_df['grna_target'] == grna_target) &
                (all_results_df['response_id_n_cell_lines'] >= min_lines) &
                (all_results_df['response_id_n_significant_cell_lines'] >= 1)
            ].copy()
    result_subset['-log10(q-value)'] = -np.log10(result_subset['FDR'])
    top_genes = result_subset.groupby('response_id')['zscore'].median().abs().sort_values(ascending=False).head(top_n).index.tolist()
    result_subset = result_subset[result_subset['response_id'].isin(top_genes)]
    result_subset['-gene_effect'] = -result_subset['gene_effect']
    sig_sbst = result_subset[result_subset['FDR'] < 0.05]
    insig_sbst = result_subset[result_subset['FDR'] >= 0.05]

    TFRC_test = run_multiple_hypergeometric(
            query_genes=top_genes,
            geneset_table=all_genesets[(all_genesets['collection'] == 'Hallmark') & (all_genesets['original_set_size'] >= 25)],
            all_genes=all_results_df['response_id'].unique(),
            report_genes=True
        )
    TFRC_test = TFRC_test.set_index('term')
    print('\nTFRC Hallmark GLYCOLYSIS top genes q =')
    print(TFRC_test.loc['HALLMARK_GLYCOLYSIS']['FDR'])

    tfrc_table = pd.concat([
        sceptre_zscore.loc['GAPDH', pd.IndexSlice[:, 'TFRC']].rename('GAPDH_z'),
        sceptre_fdr.loc['GAPDH', pd.IndexSlice[:, 'TFRC']].rename('GAPDH_FDR'),
        crispr_table[crispr_table['guide'] == 'TFRC'].set_index(['cell_line', 'guide'])['gene_effect']
    ], axis=1).rename_axis(['cell_line', 'assigned_ko']).reset_index()
    tfrc_table = tfrc_table.merge(cl_metadata[['cell_line', 'OncotreeLineage']])

    corr_result = scipy.stats.pearsonr(tfrc_table['gene_effect'], tfrc_table['GAPDH_z'])
    print('\ncorrelation between TFRC dependency and change in GAPDH expression')
    print('correlation:', corr_result)

    print(' \nignificant increase in expression was only observed in HCT15')
    print(all_results_df.query('response_id == "GAPDH" and FDR < .05'))

def ier3ip1_stats():

    gene_effect = load_file("CRISPRGeneEffect", local_dir=DOWNLOADED_DIR, index_col=0)
    gene_effect.columns = gene_effect.columns.str.split().str[0]
    profile_to_model_id = load_file('OmicsProfiles', local_dir=DOWNLOADED_DIR).query("is_default_entry").loc[
                            lambda x: x['DataType'] == 'wgs'
                            ].set_index("ProfileID")['ModelID']
    cn_seg = load_file('OmicsCNSegmentsProfileWGS', local_dir=DOWNLOADED_DIR)
    cn_seg['ModelID'] = cn_seg['ProfileID'].apply(lambda x: profile_to_model_id[x] if x in profile_to_model_id.index else pd.NA)
    cn_seg = cn_seg.rename(columns={'Chromosome': 'CONTIG',
                                    'Start': 'START',
                                    'End': 'END',
                                    'SegmentMean': 'SEGMENT_COPY_NUMBER'})
    cn_seg = cn_seg[['CONTIG', 'START', 'END', 'Status', 'SEGMENT_COPY_NUMBER', 'ModelID']]

    cytoband = load_file('cytoband', local_dir=DOWNLOADED_DIR)
    a = pybedtools.BedTool(cytoband)
    #OmicsCNSegmentsWGS
    b = pybedtools.BedTool.from_dataframe(cn_seg)
    #get intersection, outputting the overlap
    cytoband_seg_overlap = a.intersect(b, wo=True).to_dataframe(names=[
        'chrom_cytoband', 'start_cytoband', 'end_cytoband', 'name_cytoband', 'stain_cytoband', 
        'chrom', 'start', 'end', 'state', 'segment_cn', 'ModelID', 'overlap' 
    ])
    #label the cytobands
    cytoband_seg_overlap['cytoband'] = cytoband_seg_overlap['chrom_cytoband'].str.replace("chr","")+cytoband_seg_overlap['name_cytoband']
    #weight the segment cn by the length of overlap
    cytoband_seg_overlap['weighted_cn'] = cytoband_seg_overlap['segment_cn'] * cytoband_seg_overlap['overlap']
    #normalize by the total overlap
    cytoband_cn = cytoband_seg_overlap.groupby(['ModelID', 'cytoband']).agg({
        'weighted_cn': 'sum',
        'overlap': 'sum'
    }).reset_index()
    cytoband_cn['weighted_avg_cn'] = cytoband_cn['weighted_cn'] / cytoband_cn['overlap']
    cytoband_cn = cytoband_cn.set_index(['ModelID', 'cytoband'])['weighted_avg_cn'].unstack()

    plt.figure(figsize=(80 * mm, 60 * mm))
    gene_dep = gene_effect < -0.75

    cytoband_cn_renamed, gene_effect_ov = np.log2(cytoband_cn+1).align(gene_dep['IER3IP1'].dropna(), join='inner', axis=0)

    features = (
        cytoband_cn_renamed.loc[:, lambda x: x.columns.str.startswith('18q21')].mean(axis=1) < 0.75
    ).replace({True:'18q21 loss', False:None}).to_frame('18q21')
    features['11q23-4'] = (cytoband_cn_renamed.loc[:, lambda x: x.columns.str.contains('^11q2(3|4)')].mean(axis=1) < 0.75).replace({True:'11q23-4 loss', False:None})
    features['CN status'] = features.apply(lambda x: " and\n".join(x.dropna()) if x.notnull().sum() > 0 else 'neither', axis=1)
    features['CN status'] = np.where(features['CN status'].str.contains("and|neither"), features['CN status'], features['CN status']+" only")

    mwu_result = scipy.stats.mannwhitneyu(gene_effect['IER3IP1'].loc[features[features['CN status'] == 'neither'].index],
                gene_effect['IER3IP1'].loc[features[features['CN status'] == '18q21 loss and\n11q23-4 loss'].index])
    print('\nER3IP1 dependence is partially explained by hemizygous deletion of cytobands 18q21.1-32 and 11q23-4')
    print(mwu_result)

def deep_rescreen_stats():
    deep_ctrl_sig = load_file('deep_ctrl_results_sig.csv', local_dir=PROCESSED_DIR)
    for cl in ['KMRC20', 'UMRC3']:
        sig_ctrl_results = deep_ctrl_sig.query(f'cell_line == "{cl}"')
        max_expr = sig_ctrl_results.query('abs_log_2_fold_change > 1')['mean_expression'].max()
        print(f'{cl}: max expr with abs l2fc > 1: {max_expr}')

In [6]:
artifacts_stats()

Across all control knockouts in all cell lines, we found decreased expression at the targeted arm
median z-scored expression target arm: -0.02195196997879044
median z-scored expression target arm after removal of cells with aberrant position-dependent arm expression: 0.0002562061836088061

consistency of arm truncation frequency between guides targeting the same gene:
PearsonRResult(statistic=0.4606863688082777, pvalue=8.961823983927884e-64) n=1194

n cell lines with modest relationship between arm loss and avg arm dependency: 4
corr between -.25 and  -0.4132166767848523
aggregate pearson r = [-0.11584083] n= 1531
range remaning models: -0.1778242365678447, 0.1113614593470308

Median truncation frequency by TP53 status:
TP53 status
Mutant    0.143201
WT        0.125246
Name: Arm truncation frequency, dtype: float64
Mann-Whitney result on medians: TP53 mut. vs. WT
MannwhitneyuResult(statistic=20.0, pvalue=0.31318681318681324)

arm gain median frequency: 0.0373831775700934
arm loss media

  0%|          | 0/16 [00:00<?, ?it/s]/Users/nward/Desktop/perturb-seq-depmap/src/data_utils.py:107: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  table = pd.read_csv(filepath, **kwargs)
/Users/nward/Desktop/perturb-seq-depmap/src/data_utils.py:107: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  table = pd.read_csv(filepath, **kwargs)
/Users/nward/Desktop/perturb-seq-depmap/src/data_utils.py:107: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  table = pd.read_csv(filepath, **kwargs)
 19%|█▉        | 3/16 [00:00<00:00, 20.10it/s]/Users/nward/Desktop/perturb-seq-depmap/src/data_utils.py:107: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  table = pd.read_csv(filepath, **kwargs)
/Users/nward/Desktop/perturb-seq-depmap/src/data_utils.py:107: DtypeWarning: Columns (12) ha

SGO1-AS1 lower in gain, higher in loss pval= 1.6842427682229915e-05


Passing
singlet percent range: 0.23147341973680938, 0.5633004408985933
Singlet with
arm loss percent range: 0.024876240702505038, 0.12664287213940795
Multiple
infection percent range: 0.1222239847715736, 0.6419179239889637
Ambiguous
infection percent range: 0.013422485148268748, 0.4389403402972216
Unperturbed percent range: 0.0009694017051527429, 0.30472715736040606
Other
QC fail percent range: 0.02335287007685466, 0.10445748730964467
ambiguous or mixed percent range: 0.16600571065989847, 0.6606319249324074
pct arm abberation of cells otherwise passing QC: 0.10092807424593968, 0.22482295937383526
median = 0.1719428952069449


In [7]:
heterogeneity_stats()

99th quantile of index_of_dispersion = 5.187078516270499


/var/folders/3l/h4yv3dk53c74fy4gpdctdmr00000gp/T/ipykernel_99726/3464541355.py:142: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ribosomal_genes = all_genesets[(all_genesets['collection'] == 'HGNC') & (all_genesets['term'].str.contains('(R|r)ibosom'))]['gene'].tolist()


Highly variable genes: 321
Highly expressed genes: 241
Lowly expressed genes: 80
Mitochondrial genes: 10
Ribosomal genes: 77
Housekeeping genes: 121

 S100A8
Nonzero fraction of cells: 
C4I         0.320552
HKGZCC      0.009506
KYSE450     0.004788
ONE58       0.000804
RPMI7951    0.393846
SNU8        0.435897
T3M5        0.014493
Name: nonzero, dtype: float64
total: 0.14080893450045276


In [8]:
qc_stats()

is negative control pseudobulk profile most correlated?:
C4I True
HCT15 True
HKGZCC True
JVE127 True
KMRC20 True
KYSE140 True
KYSE450 True
MG63 True
ONE58 True
RPMI7951 True
SKGII True
SNU8 True
T3M5 True
U343 True
UMRC3 True
cor range in matching cell lines:  0.869431139677074 0.9456484350613956
cor range in mismatching cell lines:  -0.1918522977850236 0.8762034527142368

correlation between cell depletion and DepMap gene effect scores irrespective of cell line identity
PearsonRResult(statistic=0.6410282393897437, pvalue=3.108257789858085e-177) n= 1525
range cor within cell line:  0.4958898729651897 0.7795412813550264
median cor within cell line:  0.6580278308770917

the number of significantly downregulated targets cor with total number of UMIs recovered across all cells with the perturbation
correlation: PearsonRResult(statistic=0.8453522112014786, pvalue=3.73032879344441e-05) n= 16


Mean UMIs per cell per knockout correlation: PearsonRResult(statistic=0.45027936603976837, pvalue=0

In [9]:
global_response_stats()

observed stronger deviations from a cell line’s basal transcription state as the strength of the dependency increased
correlation: PearsonRResult(statistic=-0.6652159149608619, pvalue=3.1584362400168636e-195) n= 1523

upreg median correlation: 0.24139547721590868
downreg median correlation: 0.5645123954128154
downreg 10th percentile cor 0.4891821162381147
upreg 10th percentile cor 0.13312741132891312

n dependent knockouts = 58
term            MYC TARGETS V1  MYC TARGETS V2
term                                          
G2M CHECKPOINT        0.728380        0.623264
E2F TARGETS           0.761847        0.639155

modest upregulation of apoptotic pathways across most dependent knockouts
median z-score =
 term
APOPTOSIS                  0.260847
P53 PATHWAY                0.301759
TNFA SIGNALING VIA NFKB    0.444151
dtype: float64

interferon pathways and the apoptotic pathways showed correlated responses across dependent knockouts
mean pearson r = 0.5733968868719028

unfolded protein re

In [10]:
common_response_stats()

defined a network of gene targets using pairwise correlations 
between Z-score change in expression across our entire 
integrated dataset spanning 16 cell lines and 90 test genes, 
excluding perturbations for which we recovered fewer than 5 cells
n= 1394

XRN1 and SMG6 showed the highest degree of similarity across all pairs of cell lines
cor range: 0.1766510840107962 0.7009021225308589
median 0.4307975365199056

ADAR knockout inducing a similar response with XRN1 and SMG6
XRN1 median r 0.2309075857711964
SMG6 median r 0.21738291284348377

XRN1 and SMG6 perturbations induced an extreme upregulation of small nucleolar RNAs
grna_target
SMG6    14
XRN1    13
ADAR     2
Name: # snoRNAs with z-score > 5, dtype: int64

TFRC Hallmark GLYCOLYSIS top genes q =
6.336731430814093e-07

correlation between TFRC dependency and change in GAPDH expression
correlation: PearsonRResult(statistic=-0.46530986072334096, pvalue=0.06933309250046336)
 
ignificant increase in expression was only observed in HCT

In [11]:
ier3ip1_stats()


ER3IP1 dependence is partially explained by hemizygous deletion of cytobands 18q21.1-32 and 11q23-4
MannwhitneyuResult(statistic=14111.0, pvalue=3.277973920197136e-06)


/var/folders/3l/h4yv3dk53c74fy4gpdctdmr00000gp/T/ipykernel_99726/3464541355.py:506: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  features['11q23-4'] = (cytoband_cn_renamed.loc[:, lambda x: x.columns.str.contains('^11q2(3|4)')].mean(axis=1) < 0.75).replace({True:'11q23-4 loss', False:None})


<Figure size 314.961x236.22 with 0 Axes>

In [12]:
deep_rescreen_stats()

KMRC20: max expr with abs l2fc > 1: 0.0706874189364461
UMRC3: max expr with abs l2fc > 1: 0.0278940027894002
